# Notebook 07 — Target Construction & Autoencoder

## Censoring-Aware Target Construction and Multi-Omics Representation Learning

This notebook prepares the PRISM response targets and learns a compact
multi-omics representation for the 28 pancreatic cancer cell lines.

The PRISM response data contain extreme IC50 values and a large number
of observations for which IC50 is unavailable. Therefore, target
construction is performed before supervised model training.

At this stage, missing IC50 values are **not automatically classified
as right-censored**, because the downloaded PRISM parameter file does
not contain the administered dose range required to establish the
exact censoring boundary.

The notebook therefore:

1. Constructs a raw response-target table.
2. Diagnoses AUC as an alternative/complementary response measurement.
3. Trains a GraphTCDR-style autoencoder on GE + CNV + miRNA.
4. Produces 512-dimensional cell-line embeddings for subsequent
   GraphTCDR modelling.

## 1. Previously Completed Data Preparation

The following stages were completed before this notebook:

- 28 pancreatic cancer cell lines were matched to the GraphTCDR
  multi-omics data.
- PRISM pancreatic response data were extracted.
- 40,119 pancreatic PRISM response records were identified.
- 18,912 records contain finite positive IC50 values.
- 21,206 records have missing IC50 values.
- 1 record contains an invalid/infinite IC50 value.
- Duplicate cell–drug observations were previously aggregated to create
  the 18,037-pair integrated modelling dataset.
- 1,228 drugs were represented using 2,048-bit Morgan fingerprints.
- A fixed cold-drug split was created:
  - 982 training drugs
  - 246 test drugs
  - 0 drug overlap between train and test.

The same cold-drug split will be reused for subsequent supervised
models.

## 2. Why Target Construction Is Being Revisited

The initial response target was log10(IC50).

However, exploratory analysis showed that the IC50 distribution has an
extreme long tail:

- Minimum log10(IC50) ≈ −110.75
- Maximum log10(IC50) ≈ 287.76

Curve-fit quality also varies substantially across the response range,
with extreme IC50 values tending to have poorer R² values.

Therefore, extreme observations are **not removed automatically**.

They are retained so that we can determine whether they represent
biological responses, poorly constrained curve fits, extrapolated
estimates, or possible dose-range effects.

## 3. Censoring Decision

An earlier exploratory analysis suggested that missing IC50 values might
represent right-censored measurements.

However, the downloaded PRISM dose-response parameter file does not
contain the actual administered concentration series for each
cell–drug observation.

Therefore, an exact experimental censoring boundary cannot currently
be established from the available file.

Consequently:

- missing IC50 values are retained;
- they are not assigned an artificial IC50 value;
- they are not clipped;
- they are not labelled definitively as censored.

Instead, they are recorded as:

`right_pending_dose_check`

This preserves the censoring hypothesis for later investigation without
turning an assumption into a confirmed fact.

Build raw target table 

In [1]:
import os 
import numpy as np 
import pandas as pd

project paths

In [2]:


PROJECT = r"D:\Pancreatic_GraphTCDR"

RAW = os.path.join(
    PROJECT,
    "data",
    "raw"
)

PROC = os.path.join(
    PROJECT,
    "data",
    "processed"
)



Load the previously created pancreatic cell-line mapping

In [4]:


mapping_path = os.path.join(
    PROC,
    "pancreatic28_cell_mapping.csv"
)

if not os.path.exists(mapping_path):
    raise FileNotFoundError(
        f"Required mapping file not found:\n{mapping_path}"
    )

mapping = pd.read_csv(mapping_path)

if "cellLineName" not in mapping.columns:
    raise ValueError(
        "Expected column 'cellLineName' was not found in "
        "pancreatic28_cell_mapping.csv"
    )

pancreatic_cell_lines = (
    mapping["cellLineName"]
    .dropna()
    .unique()
    .tolist()
)

print(
    f"Pancreatic cell lines in mapping: "
    f"{len(pancreatic_cell_lines)}"
)


Pancreatic cell lines in mapping: 28


PRISM response file

In [6]:

raw_path = os.path.join(
    RAW,
    "secondary-screen-dose-response-curve-parameters.csv"
)

if not os.path.exists(raw_path):
    raise FileNotFoundError(
        f"PRISM response file not found:\n{raw_path}"
    )



 Columns required for target construction

In [7]:

use_cols = [
    "depmap_id",
    "ccle_name",
    "screen_id",
    "name",
    "ic50",
    "auc",
    "ec50",
    "r2",
    "upper_limit",
    "lower_limit",
    "slope"
]



Read the large PRISM file in chunks

In [8]:


parts = []

for chunk in pd.read_csv(
    raw_path,
    usecols=use_cols,
    chunksize=200_000
):

    matched = chunk[
        chunk["ccle_name"].isin(
            pancreatic_cell_lines
        )
    ]

    if not matched.empty:
        parts.append(matched)

if not parts:
    raise RuntimeError(
        "No pancreatic records were found in the PRISM file."
    )

df = pd.concat(
    parts,
    ignore_index=True
)



 Construct observed log10(IC50)
Only finite positive IC50 values are transformed.

In [9]:


finite_positive_ic50 = (
    np.isfinite(df["ic50"])
    & (df["ic50"] > 0)
)

df["log_ic50_observed"] = np.where(
    finite_positive_ic50,
    np.log10(df["ic50"]),
    np.nan
)



Construct conservative response status

In [10]:


df["cens_type"] = np.where(
    finite_positive_ic50,
    "observed",
    np.where(
        df["ic50"].isna(),
        "right_pending_dose_check",
        "invalid"
    )
)

# Keep AUC unchanged.
df["auc_target"] = df["auc"]

# ------------------------------------------------------------
# Print validation information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAW PANCREATIC PRISM TARGET TABLE")
print("=" * 70)

print(
    f"Total records       : {len(df):,}"
)

print(
    f"Unique cell lines   : "
    f"{df['ccle_name'].nunique():,}"
)

print(
    f"Unique drug names   : "
    f"{df['name'].nunique():,}"
)

print("\nCensoring status:")
print(
    df["cens_type"]
    .value_counts(dropna=False)
)

print("\nExpected counts:")
print("Observed finite IC50       : 18,912")
print("Missing IC50               : 21,206")
print("Invalid/infinite IC50      : 1")
print("Total                      : 40,119")

print("\nObserved log10(IC50) summary:")
print(
    df.loc[
        df["cens_type"] == "observed",
        "log_ic50_observed"
    ].describe()
)

# ------------------------------------------------------------
# Save target table
# ------------------------------------------------------------

output_path = os.path.join(
    PROC,
    "targets_raw_v1.csv"
)

df.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("TARGET TABLE SAVED")
print("=" * 70)

print(output_path)
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")


RAW PANCREATIC PRISM TARGET TABLE
Total records       : 40,119
Unique cell lines   : 28
Unique drug names   : 1,448

Censoring status:
cens_type
right_pending_dose_check    21206
observed                    18912
invalid                         1
Name: count, dtype: int64

Expected counts:
Observed finite IC50       : 18,912
Missing IC50               : 21,206
Invalid/infinite IC50      : 1
Total                      : 40,119

Observed log10(IC50) summary:
count    18912.000000
mean         0.005375
std          3.085030
min       -110.746563
25%         -0.353090
50%          0.197369
75%          0.511461
max        287.762339
Name: log_ic50_observed, dtype: float64

TARGET TABLE SAVED
D:\Pancreatic_GraphTCDR\data\processed\targets_raw_v1.csv
Rows    : 40,119
Columns : 14


## 4. AUC Diagnostic

AUC is retained as a candidate response measurement because it is available
for observations where IC50 may not be estimable.

Before selecting AUC as a modelling target, we examine:

- the overall AUC distribution;
- AUC for observations with finite IC50;
- AUC for observations without IC50;
- the proportion of AUC values ≥ 1;
- the association between log10(IC50) and AUC.

This is a diagnostic analysis only.

AUC is not assumed to be superior to IC50 before examining the results.

In [13]:


from scipy.stats import spearmanr

print("=" * 70)
print("AUC DIAGNOSTIC")
print("=" * 70)


# 1. Overall AUC summary


print("\nOverall AUC summary:")
print(df["auc"].describe())

print(
    f"\nMissing AUC: "
    f"{df['auc'].isna().sum():,}"
)

print(
    f"AUC minimum: "
    f"{df['auc'].min():.6f}"
)

print(
    f"AUC maximum: "
    f"{df['auc'].max():.6f}"
)

# ------------------------------------------------------------
# 2. AUC by IC50 status


auc_summary = (
    df.groupby("cens_type")["auc"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max"
    )
)

print("\nAUC by IC50 status:")
print(auc_summary)

# ------------------------------------------------------------
# 3. Percentage with AUC >= 1


auc_ge_1 = (
    df.groupby("cens_type")["auc"]
    .apply(
        lambda x: (x >= 1).mean() * 100
    )
)

print("\nPercentage with AUC >= 1:")
print(auc_ge_1)

# ------------------------------------------------------------
# 4. IC50-AUC association
#
# Only observations with:
# - finite positive IC50
# - finite AUC
# are included.


valid_pair = (
    np.isfinite(df["log_ic50_observed"])
    & np.isfinite(df["auc"])
)

paired = df.loc[
    valid_pair,
    ["log_ic50_observed", "auc"]
].copy()

rho, p_value = spearmanr(
    paired["log_ic50_observed"],
    paired["auc"]
)

print("\nIC50-AUC association:")
print(
    f"Valid paired observations : "
    f"{len(paired):,}"
)

print(
    f"Spearman rho              : "
    f"{rho:.6f}"
)

print(
    f"P-value                   : "
    f"{p_value:.6e}"
)

# ------------------------------------------------------------
# 5. Median AUC comparison


observed_auc = df.loc[
    df["cens_type"] == "observed",
    "auc"
].dropna()

missing_auc = df.loc[
    df["cens_type"] == "right_pending_dose_check",
    "auc"
].dropna()

print("\nMedian AUC comparison:")

print(
    f"Observed IC50 : "
    f"{observed_auc.median():.6f}"
)

print(
    f"Missing IC50  : "
    f"{missing_auc.median():.6f}"
)

AUC DIAGNOSTIC

Overall AUC summary:
count    40119.000000
mean         0.978968
std          0.283558
min          0.005819
25%          0.824568
50%          0.930470
75%          1.158616
max          3.244968
Name: auc, dtype: float64

Missing AUC: 0
AUC minimum: 0.005819
AUC maximum: 3.244968

AUC by IC50 status:
                          count      mean    median       std   minimum  \
cens_type                                                                 
invalid                       1  0.636938  0.636938       NaN  0.636938   
observed                  18912  0.775462  0.826983  0.166090  0.005819   
right_pending_dose_check  21206  1.160476  1.141015  0.240061  0.576518   

                           maximum  
cens_type                           
invalid                   0.636938  
observed                  1.000000  
right_pending_dose_check  3.244968  

Percentage with AUC >= 1:
cens_type
invalid                      0.000000
observed                     0.005288
right_

##  AUC Diagnostic — Initial Findings

The AUC diagnostic shows that AUC is available for all 40,119 pancreatic
PRISM response records.

The median AUC differs substantially between observations with and without
finite IC50:

- Observed finite IC50: median AUC ≈ 0.827
- Missing IC50: median AUC ≈ 1.141

The missing-IC50 group therefore shows a higher AUC distribution than the
observed-IC50 group.

This indicates that missing IC50 values are associated with a different
response regime and provides evidence that the missingness is not simply
an arbitrary loss of observations.

However, this result does **not** by itself prove right-censoring.

The actual experimental dose range is still unavailable from the current
PRISM parameter file. Therefore, the missing-IC50 observations remain
labelled `right_pending_dose_check`.

The relationship between IC50 and AUC will be examined next using the
Spearman correlation calculated above.

In [14]:
# ============================================================
# Display the IC50-AUC association clearly
# ============================================================

valid_pair = (
    np.isfinite(df["log_ic50_observed"])
    & np.isfinite(df["auc"])
)

paired = df.loc[
    valid_pair,
    ["log_ic50_observed", "auc"]
].copy()

rho, p_value = spearmanr(
    paired["log_ic50_observed"],
    paired["auc"]
)

print("=" * 70)
print("IC50–AUC ASSOCIATION")
print("=" * 70)

print(f"Valid paired observations : {len(paired):,}")
print(f"Spearman rho              : {rho:.6f}")
print(f"P-value                   : {p_value:.6e}")

print("\nInterpretation:")
print(
    "The correlation measures whether higher log10(IC50) values "
    "tend to be associated with higher or lower AUC."
)

IC50–AUC ASSOCIATION
Valid paired observations : 18,912
Spearman rho              : 0.915239
P-value                   : 0.000000e+00

Interpretation:
The correlation measures whether higher log10(IC50) values tend to be associated with higher or lower AUC.


## Interpretation of the IC50–AUC Relationship

The Spearman correlation between observed log10(IC50) and AUC is:

**ρ = 0.915239**

across 18,912 observations with both measurements available.

This indicates a strong positive monotonic association between the two
response measurements.

Therefore, AUC captures response information that is strongly related to
the IC50 response where IC50 can be estimated.

At the same time, AUC is available for all 40,119 observations,
including the 21,206 observations without finite IC50.

This makes AUC a potentially valuable complementary response measurement.

However, the strong correlation does not mean that AUC and IC50 are
interchangeable. They represent different summaries of the dose-response
curve.

The next step is therefore to examine whether the extreme IC50 values
also show disagreement with EC50 and/or poor curve-fit quality.

This is important because the central research question concerns whether
the extreme IC50 tail represents reliable biological information or
poorly constrained response estimates.

In [15]:
# ============================================================
# Cell 4 — EC50–IC50 Consistency Diagnostic
# ============================================================

print("=" * 70)
print("EC50–IC50 CONSISTENCY DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# Keep only finite positive IC50 and EC50 values
# ------------------------------------------------------------

valid_ec50_ic50 = (
    np.isfinite(df["ic50"])
    & (df["ic50"] > 0)
    & np.isfinite(df["ec50"])
    & (df["ec50"] > 0)
)

ec50_ic50 = df.loc[
    valid_ec50_ic50,
    [
        "log_ic50_observed",
        "ec50",
        "r2",
        "auc"
    ]
].copy()

ec50_ic50["log_ec50"] = np.log10(
    ec50_ic50["ec50"]
)

ec50_ic50["ec50_ic50_difference"] = (
    ec50_ic50["log_ic50_observed"]
    - ec50_ic50["log_ec50"]
)

ec50_ic50["absolute_difference"] = (
    ec50_ic50["ec50_ic50_difference"]
    .abs()
)

# ------------------------------------------------------------
# Define the same IC50 ranges used in the previous analysis
# ------------------------------------------------------------

bins = [
    -np.inf,
    -3,
    -2,
    -1,
    0,
    1,
    2,
    3,
    np.inf
]

labels = [
    "< -3",
    "-3 to -2",
    "-2 to -1",
    "-1 to 0",
    "0 to 1",
    "1 to 2",
    "2 to 3",
    ">= 3"
]

ec50_ic50["ic50_range"] = pd.cut(
    ec50_ic50["log_ic50_observed"],
    bins=bins,
    labels=labels,
    right=False
)

# ------------------------------------------------------------
# Summarise agreement across the IC50 distribution
# ------------------------------------------------------------

summary = (
    ec50_ic50
    .groupby(
        "ic50_range",
        observed=False
    )
    .agg(
        N=("absolute_difference", "size"),
        median_log_ic50=(
            "log_ic50_observed",
            "median"
        ),
        median_log_ec50=(
            "log_ec50",
            "median"
        ),
        median_difference=(
            "ec50_ic50_difference",
            "median"
        ),
        median_absolute_difference=(
            "absolute_difference",
            "median"
        ),
        Q95_absolute_difference=(
            "absolute_difference",
            lambda x: x.quantile(0.95)
        ),
        median_r2=("r2", "median")
    )
)

print("\nEC50–IC50 agreement by IC50 range:")
print(summary.to_string())

# ------------------------------------------------------------
# Overall agreement
# ------------------------------------------------------------

print("\nOverall:")
print(
    f"Valid EC50–IC50 observations: "
    f"{len(ec50_ic50):,}"
)

print(
    f"Median absolute difference: "
    f"{ec50_ic50['absolute_difference'].median():.6f}"
)

print(
    f"Mean absolute difference: "
    f"{ec50_ic50['absolute_difference'].mean():.6f}"
)

print(
    f"95th percentile absolute difference: "
    f"{ec50_ic50['absolute_difference'].quantile(0.95):.6f}"
)

EC50–IC50 CONSISTENCY DIAGNOSTIC

EC50–IC50 agreement by IC50 range:
                N  median_log_ic50  median_log_ec50  median_difference  median_absolute_difference  Q95_absolute_difference  median_r2
ic50_range                                                                                                                            
< -3          225        -3.711643        -3.285078           0.000325                    0.096190                 9.542450   0.108885
-3 to -2      428        -2.346655        -2.380691           0.023399                    0.030358                 0.367906   0.766693
-2 to -1     1615        -1.425149        -1.479702           0.029496                    0.034422                 0.414272   0.738775
-1 to 0      5240        -0.330522        -0.389042           0.024126                    0.028888                 0.333247   0.608689
0 to 1      11082         0.436568         0.374319           0.021365                    0.026785                 0.3137

## Curve-Fit Quality Across the IC50 Distribution

The previous analysis showed that IC50–EC50 disagreement increases sharply
in the upper response tail.

We now quantify curve-fit quality directly using R².

For each IC50 range we calculate:

- number of observations;
- median R²;
- mean R²;
- percentage with negative R²;
- percentage with R² < 0.2.

These statistics are descriptive and are not used to remove observations.

The purpose is to determine whether poor curve-fit quality is concentrated
in the extreme response regions.

In [16]:
# ============================================================
# Curve-Fit Quality Across IC50 Distribution
# ============================================================

print("=" * 70)
print("CURVE-FIT QUALITY ACROSS IC50 DISTRIBUTION")
print("=" * 70)

# ------------------------------------------------------------
# Work only with observed finite positive IC50 values
# ------------------------------------------------------------

quality = df.loc[
    df["cens_type"] == "observed",
    ["log_ic50_observed", "r2"]
].copy()

# ------------------------------------------------------------
# Same IC50 ranges used throughout the analysis
# ------------------------------------------------------------

bins = [
    -np.inf,
    -3,
    -2,
    -1,
    0,
    1,
    2,
    3,
    np.inf
]

labels = [
    "< -3",
    "-3 to -2",
    "-2 to -1",
    "-1 to 0",
    "0 to 1",
    "1 to 2",
    "2 to 3",
    ">= 3"
]

quality["ic50_range"] = pd.cut(
    quality["log_ic50_observed"],
    bins=bins,
    labels=labels,
    right=False
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

quality_summary = (
    quality
    .groupby(
        "ic50_range",
        observed=False
    )
    .agg(
        N=("r2", "count"),
        median_R2=("r2", "median"),
        mean_R2=("r2", "mean"),
        negative_R2_pct=(
            "r2",
            lambda x: (x < 0).mean() * 100
        ),
        R2_below_02_pct=(
            "r2",
            lambda x: (x < 0.2).mean() * 100
        )
    )
)

print("\nCurve-fit quality by IC50 range:")
print(
    quality_summary.to_string()
)

# ------------------------------------------------------------
# Overall quality
# ------------------------------------------------------------

print("\nOverall observed IC50 quality:")

print(
    f"Median R² : "
    f"{quality['r2'].median():.6f}"
)

print(
    f"Mean R²   : "
    f"{quality['r2'].mean():.6f}"
)

print(
    f"Negative R² : "
    f"{(quality['r2'] < 0).mean() * 100:.3f}%"
)

print(
    f"R² < 0.2   : "
    f"{(quality['r2'] < 0.2).mean() * 100:.3f}%"
)

CURVE-FIT QUALITY ACROSS IC50 DISTRIBUTION

Curve-fit quality by IC50 range:
                N  median_R2   mean_R2  negative_R2_pct  R2_below_02_pct
ic50_range                                                              
< -3          225   0.108885  0.226843        34.222222        54.222222
-3 to -2      428   0.766693  0.693731         2.803738         6.308411
-2 to -1     1615   0.738775  0.690050         1.114551         3.467492
-1 to 0      5240   0.608689  0.576298         1.698473         6.507634
0 to 1      11082   0.411352  0.379779         8.942429        22.613247
1 to 2        216   0.123736  0.156980        25.925926        58.333333
2 to 3         34   0.001938 -0.421490        50.000000        73.529412
>= 3           72  -0.012675 -0.041088        61.111111        97.222222

Overall observed IC50 quality:
Median R² : 0.494767
Mean R²   : 0.460423
Negative R² : 6.895%
R² < 0.2   : 17.306%


##  Response-Target Decision

The response diagnostics provide three complementary findings.

### Finding 1 — IC50 and AUC are strongly related

The Spearman correlation between log10(IC50) and AUC is:

**ρ = 0.915239**

This indicates that AUC captures response information strongly associated
with IC50 for observations where IC50 is available.

### Finding 2 — AUC is available across the complete PRISM response table

AUC is available for all:

**40,119 / 40,119 observations**

including the 21,206 observations without finite IC50.

The median AUC is:

- 0.827 for observations with finite IC50
- 1.141 for observations without IC50

Thus, the observations without IC50 represent a systematically different
response regime rather than simply a random subset of the data.

### Finding 3 — Extreme IC50 values have poor curve-fit quality

The upper IC50 tail shows rapidly decreasing R² and increasing
disagreement between IC50 and EC50.

For log10(IC50) ≥ 3:

- median R² ≈ −0.013
- 61.1% have negative R²
- 97.2% have R² < 0.2
- median absolute IC50–EC50 disagreement ≈ 5.53 log units

Therefore, the extreme upper IC50 values should not be treated as
equally reliable measurements without further investigation.

---

### Target strategy

Based on these diagnostics, the project will use:

**Primary response target: AUC**

AUC is selected for the main response-prediction experiment because it is
available across the complete PRISM response table and is strongly
associated with the observed IC50 response.

**Secondary response target: observed log10(IC50)**

Finite positive log10(IC50) will be retained as a secondary target so
that the project can directly compare its behaviour with the AUC model.

**Censoring-aware IC50: future experimental analysis**

Missing IC50 observations will not be assigned an artificial censoring
value at this stage.

The censoring hypothesis remains a separate research question because the
exact experimental dose boundaries are not available in the current
parameter file.

---

### Important modelling principle

The extreme IC50 observations are **not deleted**.

Instead, their poor curve-fit quality is retained as an important
diagnostic variable (`R²`) for later quality-aware modelling.

Therefore, the project can test whether explicitly incorporating
response quality improves prediction without assuming beforehand that
quality-aware modelling will improve performance.

##  Phase 6 Target-Diagnostic Conclusion

The response-data investigation supports the following conclusion:

> PRISM pancreatic response measurements contain a substantial IC50
> long tail whose extreme upper region is associated with poor curve-fit
> quality and strong IC50–EC50 disagreement.

At the same time, AUC provides a complete response measurement across the
40,119-record pancreatic PRISM subset and shows a strong monotonic
relationship with observed IC50.

Therefore, the next modelling stage will use AUC as the main response
target while retaining observed log10(IC50) and curve-fit R² for
secondary analysis.

The analysis does not claim that extreme IC50 values are artifacts or
that missing IC50 values are definitively right-censored.

Those remain hypotheses to be tested experimentally.

### Decision

**Measure → Diagnose → Select AUC → Retain IC50/R² for secondary and
quality-aware analysis → Proceed to representation learning.**

## Audit of IC50-Missing Records Using AUC and Curve-Fit Parameters

### Objective

Before selecting the final response target for the GraphTCDR experiments, we need to understand the records for which IC50 is missing or classified as `right_pending_dose_check`.

The previous analysis showed that AUC is available for a much larger number of PRISM response records than IC50. Therefore, we compare the **IC50-present** and **IC50-missing / right-pending-dose-check** groups using AUC and the fitted lower asymptote.

### What we are checking

For both groups, we calculate:

- Number of records (`N`)
- Median AUC
- Mean AUC
- Percentage of records with AUC ≥ 1
- Percentage with `lower_limit > 0.5`
- Percentage with missing or non-finite AUC

### Why these checks matter

AUC summarizes the response across the tested concentration range and is available even when an IC50 value cannot be obtained.

An AUC ≥ 1 may indicate weak or absent growth inhibition relative to the control, while a high fitted lower asymptote may indicate that the response curve does not reach the expected inhibition range.

Therefore, if the IC50-missing group shows systematically higher AUC values and different curve-fit characteristics than the IC50-present group, this provides evidence that the missing IC50 values may represent biologically meaningful resistant/no-inhibition responses rather than simply random failed fits.

However, **this analysis does not by itself prove that the observations are censored**. The exact tested dose range is not available in the current response-parameter file, so any censoring interpretation must remain cautious.

### Expected outcome

This audit will provide evidence for deciding whether **AUC should be used as the primary response target** in the first controlled GraphTCDR experiment, while retaining `log10(IC50)` as a secondary target for comparison.

In [1]:
import os
import pandas as pd
import numpy as np

# Project paths
ROOT = r"D:\Pancreatic_GraphTCDR"
PROC = os.path.join(ROOT, "data", "processed")

# Load the target table
target_path = os.path.join(PROC, "targets_raw_v1.csv")
d = pd.read_csv(target_path)

print("Target table shape:", d.shape)
print("Columns:", list(d.columns))

# Identify the no-IC50 group using the project's existing classification
if "cens_type" in d.columns:
    no_ic50 = d[d["cens_type"] == "right_pending_dose_check"].copy()
else:
    # Fallback only if cens_type is absent
    no_ic50 = d[d["ic50"].isna() | ~np.isfinite(d["ic50"])].copy()

ic50_present = d[~d.index.isin(no_ic50.index)].copy()

print("\n" + "=" * 60)
print("NO-IC50 / RIGHT-PENDING-DOSE-CHECK GROUP")
print("=" * 60)

print("N:", len(no_ic50))
print("Median AUC:", no_ic50["auc"].median())
print("Mean AUC:", no_ic50["auc"].mean())
print("% AUC >= 1:", round(100 * (no_ic50["auc"] >= 1).mean(), 2))
print(
    "% lower_limit > 0.5:",
    round(100 * (no_ic50["lower_limit"] > 0.5).mean(), 2)
)
print(
    "% AUC missing/non-finite:",
    round(
        100 * (
            no_ic50["auc"].isna()
            | ~np.isfinite(no_ic50["auc"])
        ).mean(),
        2
    )
)

print("\n" + "=" * 60)
print("IC50-PRESENT GROUP")
print("=" * 60)

print("N:", len(ic50_present))
print("Median AUC:", ic50_present["auc"].median())
print("Mean AUC:", ic50_present["auc"].mean())
print("% AUC >= 1:", round(100 * (ic50_present["auc"] >= 1).mean(), 2))
print(
    "% lower_limit > 0.5:",
    round(100 * (ic50_present["lower_limit"] > 0.5).mean(), 2)
)
print(
    "% AUC missing/non-finite:",
    round(
        100 * (
            ic50_present["auc"].isna()
            | ~np.isfinite(ic50_present["auc"])
        ).mean(),
        2
    )
)

Target table shape: (40119, 14)
Columns: ['depmap_id', 'ccle_name', 'screen_id', 'upper_limit', 'lower_limit', 'slope', 'r2', 'auc', 'ec50', 'ic50', 'name', 'log_ic50_observed', 'cens_type', 'auc_target']

NO-IC50 / RIGHT-PENDING-DOSE-CHECK GROUP
N: 21206
Median AUC: 1.1410149643269651
Mean AUC: 1.1604755295420313
% AUC >= 1: 71.42
% lower_limit > 0.5: 99.38
% AUC missing/non-finite: 0.0

IC50-PRESENT GROUP
N: 18913
Median AUC: 0.826980743260684
Mean AUC: 0.7754546122217674
% AUC >= 1: 0.01
% lower_limit > 0.5: 0.0
% AUC missing/non-finite: 0.0
